In [1]:
import torch
import math
import torch.nn.functional as F

In [6]:
batch_size = 2
seq_len = 9
n_heads = 2
dhead = 6
m = int(math.sqrt(seq_len))
dhead_half = dhead // 2

top_k = 2

In [7]:
k = torch.rand(batch_size, n_heads, m, m, dhead)
q = torch.rand(batch_size, n_heads, seq_len, dhead)
v = torch.rand(batch_size, n_heads, seq_len, dhead)

In [8]:
def __get_topk_candidates(query, key):
    scores = torch.matmul(query, key.transpose(-2, -1))
    
    _, indices = torch.topk(scores, k=top_k, dim=-1)
    
    key_expanded = key.unsqueeze(2).expand(-1, -1, seq_len, -1, -1)
    ind_expanded = indices.unsqueeze(-1).expand(-1, -1, -1, -1, dhead_half)
    selected_vecs = torch.gather(key_expanded, 3, ind_expanded)
    
    return selected_vecs, indices

def __gather_selected(source_tensor, idx_tensor):
    # source: (..., Num_Candidates, D)
    # idx:    (..., K) -> expand to (..., K, D)
    idx_expanded = idx_tensor.unsqueeze(-1).expand(-1, -1, -1, -1, source_tensor.size(-1))
    return torch.gather(source_tensor, 3, idx_expanded)

In [9]:
k1 = k[..., : dhead_half].sum(-2)
k2 = k[..., dhead_half :].sum(-3)

q1 = q[..., : dhead_half]
q2 = q[..., dhead_half :]

k1_vecs, k1_idxs = __get_topk_candidates(q1, k1)
k2_vecs, k2_idxs = __get_topk_candidates(q2, k2)

c1 = k1_vecs.unsqueeze(-2).expand(-1, -1, -1, top_k, top_k, dhead_half)
c2 = k2_vecs.unsqueeze(-3).expand(-1, -1, -1, top_k, top_k, dhead_half)
candidates = torch.cat([c1, c2], dim=-1).view(batch_size, n_heads, seq_len, -1, dhead)

# Second Retrieval (Find closest among combinations)
scores_final = (q.unsqueeze(-2) * candidates).sum(dim=-1)

_, selection_indices = torch.topk(scores_final, k=top_k, dim=-1)

final_k = __gather_selected(candidates, selection_indices)

idx_in_k1 = selection_indices // top_k
idx_in_k2 = selection_indices % top_k

final_row_idxs = torch.gather(k1_idxs, 3, idx_in_k1)
final_col_idxs = torch.gather(k2_idxs, 3, idx_in_k2)

v_indices = (final_row_idxs * m) + final_col_idxs
v_flat_exp = v.unsqueeze(2).expand(-1, -1, seq_len, -1, -1)
final_v = __gather_selected(v_flat_exp, v_indices)

# --- Attention: Softmax(Q @ K.T) @ V ---
# ! consider dividing by sqrt(dhead) or smth else as we have less vectors in multiplication?
# ! divide by sqrt(self.top_k) ?
attn_scores = torch.matmul(q.unsqueeze(-2), final_k.transpose(-2, -1))

# no causal mask as we work in encoder-only setting
# causal_mask = torch.triu(
#     torch.ones(seq_len, seq_len, device=attn_scores.device), diagonal=1
# ).bool()
# attn_scores = attn_scores.masked_fill(causal_mask, float("-inf"))

attn_weights = F.softmax(attn_scores, dim=-1)

attn_output = torch.matmul(attn_weights, final_v)
attn_output = attn_output.squeeze(-2)

In [18]:
k1_vecs.shape

torch.Size([2, 2, 9, 2, 3])

In [19]:
k1_vecs[0,0,...], k1_idxs[0,0,...]

(tensor([[[1.2311, 2.5849, 1.5339],
          [0.8378, 1.1525, 2.0385]],
 
         [[1.2311, 2.5849, 1.5339],
          [0.9238, 2.2088, 1.3387]],
 
         [[1.2311, 2.5849, 1.5339],
          [0.8378, 1.1525, 2.0385]],
 
         [[1.2311, 2.5849, 1.5339],
          [0.8378, 1.1525, 2.0385]],
 
         [[1.2311, 2.5849, 1.5339],
          [0.9238, 2.2088, 1.3387]],
 
         [[1.2311, 2.5849, 1.5339],
          [0.8378, 1.1525, 2.0385]],
 
         [[1.2311, 2.5849, 1.5339],
          [0.9238, 2.2088, 1.3387]],
 
         [[1.2311, 2.5849, 1.5339],
          [0.9238, 2.2088, 1.3387]],
 
         [[1.2311, 2.5849, 1.5339],
          [0.9238, 2.2088, 1.3387]]]),
 tensor([[0, 1],
         [0, 2],
         [0, 1],
         [0, 1],
         [0, 2],
         [0, 1],
         [0, 2],
         [0, 2],
         [0, 2]]))

In [12]:
before_out_proj = attn_output.transpose(1, 2).contiguous().flatten(-2)

In [ ]:
before_out_proj.shape  # should be (batch_size, seq_len, n_heads * dhead)

torch.Size([2, 9, 12])

In [14]:
(batch_size, seq_len, n_heads * dhead)

(2, 9, 12)